In [182]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import statistics
from math import sqrt
matplotlib.use("Agg")
sns.set()

from simulate import generate_matrices_orthogonal, sim_data, collect_precision_matrix
from likelihood import full_likelihood, likelihood_ratio_test
from numpy.linalg import inv as inv
from optim import optim_boyd, optim_boyd_dc
from utils import symmetrize_from_vector
np.random.seed(42)


In [183]:
def plot_confidence_interval(x, values, z=1.96, color='#2187bb', horizontal_line_width=0.25, constrain=False):
    mean = statistics.mean(values)
    stdev = statistics.stdev(values)
    confidence_interval = z * stdev / sqrt(len(values))

    left = x - horizontal_line_width / 2
    top = mean - confidence_interval
    right = x + horizontal_line_width / 2
    bottom = mean + confidence_interval
    if constrain: # for pvalue plots
        if top <= 0.0:
            top = 0.0
    plt.plot([x, x], [top, bottom], color=color)
    plt.plot([left, right], [top, top], color=color)
    plt.plot([left, right], [bottom, bottom], color=color)
    plt.plot(x, mean, 'o', color='#f44336')

    return mean, confidence_interval

In [184]:
M = 4
dim = 40
H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  2  3 11 12 14 25 30 32]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  8 19 20 21 22]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 4 10 23 28 34 35]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [ 5  6  7  9 13 15 16 17 18 24 26 27 29 31 33 36 37 38 39]
****************************************



In [185]:
prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
second_prec_coeffs = np.ones(M)
second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.1
prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)

In [186]:
data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
data_full = np.concatenate((data_one, data_two), axis=1)

In [187]:
data_one.shape, data_two.shape, data_full.shape

((40, 100), (40, 100), (40, 200))

In [188]:
C_one = np.cov(data_one, bias=True)
C_two = np.cov(data_two, bias=True)
C_full = np.cov(data_full, bias=True)

In [189]:
coeffs_hat_total = optim_boyd(C=C_full, H_s=H_s)
alpha_i_change_pre = coeffs_hat_total.copy()
alpha_i_change_post = coeffs_hat_total.copy()
curr_alpha_i_pre = optim_boyd_dc(C=C_one, H=H_s[-1])
curr_alpha_i_post = optim_boyd_dc(C=C_two, H=H_s[-1])
alpha_i_change_pre[-1] = curr_alpha_i_pre
alpha_i_change_post[-1] = curr_alpha_i_post

In [190]:
null_likelihood = full_likelihood(coeffs_hat_total, H_s, C_full, N=data_full.shape[1], 
                                          lam=5e-2, include_l1=False, debug_title='global')
# # likelihood on pre data, alpha_one change
alt_likelihood_alpha_i_pre = full_likelihood(alpha_i_change_pre, H_s, C_one, N=data_one.shape[1], 
                                                lam=5e-2, include_l1=False, debug_title='Pre')
# likelihood on post data, alpha_one change
alt_likelihood_alpha_i_post = full_likelihood(alpha_i_change_post, H_s, C_two, N=data_two.shape[1], 
                                                lam=5e-2, include_l1=False, debug_title='Post')

In [191]:
alt_likelihood_alpha_i = alt_likelihood_alpha_i_pre + alt_likelihood_alpha_i_post
dof = 2
test_stat_i, p_val_i = likelihood_ratio_test(null_likelihood, 
                                    alt_likelihood_alpha_i, dof, log_pvals=0)
curr_mat = symmetrize_from_vector(H_s[-1], dim)
nonzero_cols = np.nonzero(np.any(curr_mat != 0, axis=0))[0]
C_val = len(nonzero_cols)
if C_val > 0:
    correction_factor = dim/C_val
    print(correction_factor)
    # correct p_vals for cluster at all time points independently
    p_val_i = p_val_i*correction_factor
else:
    p_val_i = 1.0

2.1052631578947367


In [192]:
p_val_i

8.731771169264175e-11

In [193]:
coeffs_hat_total[-1], alpha_i_change_pre[-1], alpha_i_change_post[-1]

(0.9184011105065961, 1.0271608157324528, 0.8344819386452029)

In [194]:
alpha_i_change_pre[-1]-alpha_i_change_post[-1]

0.19267887708724984

In [195]:
seed_list = np.arange(0, 100)
coeff_diffs_opo = []
p_vals_opo = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.1
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    C_one = np.cov(data_one, bias=True)
    C_two = np.cov(data_two, bias=True)
    C_full = np.cov(data_full, bias=True)
    coeffs_hat_total = optim_boyd(C=C_full, H_s=H_s)
    alpha_i_change_pre = coeffs_hat_total.copy()
    alpha_i_change_post = coeffs_hat_total.copy()
    curr_alpha_i_pre = optim_boyd_dc(C=C_one, H=H_s[-1])
    curr_alpha_i_post = optim_boyd_dc(C=C_two, H=H_s[-1])
    alpha_i_change_pre[-1] = curr_alpha_i_pre
    alpha_i_change_post[-1] = curr_alpha_i_post
    null_likelihood = full_likelihood(coeffs_hat_total, H_s, C_full, N=data_full.shape[1], 
                                          lam=5e-2, include_l1=False, debug_title='global')
    # # likelihood on pre data, alpha_one change
    alt_likelihood_alpha_i_pre = full_likelihood(alpha_i_change_pre, H_s, C_one, N=data_one.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Pre')
    # likelihood on post data, alpha_one change
    alt_likelihood_alpha_i_post = full_likelihood(alpha_i_change_post, H_s, C_two, N=data_two.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Post')
    alt_likelihood_alpha_i = alt_likelihood_alpha_i_pre + alt_likelihood_alpha_i_post
    dof = 2
    test_stat_i, p_val_i = likelihood_ratio_test(null_likelihood, 
                                        alt_likelihood_alpha_i, dof, log_pvals=0)
    curr_mat = symmetrize_from_vector(H_s[-1], dim)
    nonzero_cols = np.nonzero(np.any(curr_mat != 0, axis=0))[0]
    C_val = len(nonzero_cols)
    if C_val > 0:
        correction_factor = dim/C_val
        print(correction_factor)
        # correct p_vals for cluster at all time points independently
        p_val_i = p_val_i*correction_factor
    else:
        p_val_i = 1.0
    p_vals_opo.append(p_val_i)
    coeff_diffs_opo.append(alpha_i_change_pre[-1]-alpha_i_change_post[-1])
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  3  4  8 11 12 14 15 16 17 18 19 20 22 24 28 32 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  2  5  7 10 13 21 23 25 26 30 31 33 34 35 36 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 6  9 27 37]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [29]
****************************************

40.0
****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  1  4  9 10 17 18 19 21 22 25 28 32]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Containe

In [196]:
seed_list = np.arange(0, 100)
coeff_diffs_opf = []
p_vals_opf = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.4
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    C_one = np.cov(data_one, bias=True)
    C_two = np.cov(data_two, bias=True)
    C_full = np.cov(data_full, bias=True)
    coeffs_hat_total = optim_boyd(C=C_full, H_s=H_s)
    alpha_i_change_pre = coeffs_hat_total.copy()
    alpha_i_change_post = coeffs_hat_total.copy()
    curr_alpha_i_pre = optim_boyd_dc(C=C_one, H=H_s[-1])
    curr_alpha_i_post = optim_boyd_dc(C=C_two, H=H_s[-1])
    alpha_i_change_pre[-1] = curr_alpha_i_pre
    alpha_i_change_post[-1] = curr_alpha_i_post
    null_likelihood = full_likelihood(coeffs_hat_total, H_s, C_full, N=data_full.shape[1], 
                                          lam=5e-2, include_l1=False, debug_title='global')
    # # likelihood on pre data, alpha_one change
    alt_likelihood_alpha_i_pre = full_likelihood(alpha_i_change_pre, H_s, C_one, N=data_one.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Pre')
    # likelihood on post data, alpha_one change
    alt_likelihood_alpha_i_post = full_likelihood(alpha_i_change_post, H_s, C_two, N=data_two.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Post')
    alt_likelihood_alpha_i = alt_likelihood_alpha_i_pre + alt_likelihood_alpha_i_post
    dof = 2
    test_stat_i, p_val_i = likelihood_ratio_test(null_likelihood, 
                                        alt_likelihood_alpha_i, dof, log_pvals=0)
    curr_mat = symmetrize_from_vector(H_s[-1], dim)
    nonzero_cols = np.nonzero(np.any(curr_mat != 0, axis=0))[0]
    C_val = len(nonzero_cols)
    if C_val > 0:
        correction_factor = dim/C_val
        print(correction_factor)
        # correct p_vals for cluster at all time points independently
        p_val_i = p_val_i*correction_factor
    else:
        p_val_i = 1.0
    p_vals_opf.append(p_val_i)
    coeff_diffs_opf.append(alpha_i_change_pre[-1]-alpha_i_change_post[-1])
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  4 11 12 17 26 29 30 33 35]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  2  3  7  8  9 10 13 16 19 21 22 23 25 27 28 31 34 36 37]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 5  6 14 32 38 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [15 18 20 24]
****************************************

10.0
****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  5 13 16 17 20 21 23 36 37 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1 

In [197]:
seed_list = np.arange(0, 100)
coeff_diffs_ope = []
p_vals_ope = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.8
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    C_one = np.cov(data_one, bias=True)
    C_two = np.cov(data_two, bias=True)
    C_full = np.cov(data_full, bias=True)
    coeffs_hat_total = optim_boyd(C=C_full, H_s=H_s)
    alpha_i_change_pre = coeffs_hat_total.copy()
    alpha_i_change_post = coeffs_hat_total.copy()
    curr_alpha_i_pre = optim_boyd_dc(C=C_one, H=H_s[-1])
    curr_alpha_i_post = optim_boyd_dc(C=C_two, H=H_s[-1])
    alpha_i_change_pre[-1] = curr_alpha_i_pre
    alpha_i_change_post[-1] = curr_alpha_i_post
    null_likelihood = full_likelihood(coeffs_hat_total, H_s, C_full, N=data_full.shape[1], 
                                          lam=5e-2, include_l1=False, debug_title='global')
    # # likelihood on pre data, alpha_one change
    alt_likelihood_alpha_i_pre = full_likelihood(alpha_i_change_pre, H_s, C_one, N=data_one.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Pre')
    # likelihood on post data, alpha_one change
    alt_likelihood_alpha_i_post = full_likelihood(alpha_i_change_post, H_s, C_two, N=data_two.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Post')
    alt_likelihood_alpha_i = alt_likelihood_alpha_i_pre + alt_likelihood_alpha_i_post
    dof = 2
    test_stat_i, p_val_i = likelihood_ratio_test(null_likelihood, 
                                        alt_likelihood_alpha_i, dof, log_pvals=0)
    curr_mat = symmetrize_from_vector(H_s[-1], dim)
    nonzero_cols = np.nonzero(np.any(curr_mat != 0, axis=0))[0]
    C_val = len(nonzero_cols)
    if C_val > 0:
        correction_factor = dim/C_val
        print(correction_factor)
        # correct p_vals for cluster at all time points independently
        p_val_i = p_val_i*correction_factor
    else:
        p_val_i = 1.0
    p_vals_ope.append(p_val_i)
    coeff_diffs_ope.append(alpha_i_change_pre[-1]-alpha_i_change_post[-1])
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  2  5  8  9 13 14 18 21 24 34 37 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  3  7 19 23 28]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [4]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [ 6 10 11 12 15 16 17 20 22 25 26 27 29 30 31 32 33 35 36 39]
****************************************

2.0
****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  1  6  7  8 12 16 22 23 24 25 29 36 37]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contain

In [198]:
seed_list = np.arange(0, 100)
coeff_diffs_zero = []
p_vals_zero = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.0
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    C_one = np.cov(data_one, bias=True)
    C_two = np.cov(data_two, bias=True)
    C_full = np.cov(data_full, bias=True)
    coeffs_hat_total = optim_boyd(C=C_full, H_s=H_s)
    alpha_i_change_pre = coeffs_hat_total.copy()
    alpha_i_change_post = coeffs_hat_total.copy()
    curr_alpha_i_pre = optim_boyd_dc(C=C_one, H=H_s[-1])
    curr_alpha_i_post = optim_boyd_dc(C=C_two, H=H_s[-1])
    alpha_i_change_pre[-1] = curr_alpha_i_pre
    alpha_i_change_post[-1] = curr_alpha_i_post
    null_likelihood = full_likelihood(coeffs_hat_total, H_s, C_full, N=data_full.shape[1], 
                                          lam=5e-2, include_l1=False, debug_title='global')
    # # likelihood on pre data, alpha_one change
    alt_likelihood_alpha_i_pre = full_likelihood(alpha_i_change_pre, H_s, C_one, N=data_one.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Pre')
    # likelihood on post data, alpha_one change
    alt_likelihood_alpha_i_post = full_likelihood(alpha_i_change_post, H_s, C_two, N=data_two.shape[1], 
                                                    lam=5e-2, include_l1=False, debug_title='Post')
    alt_likelihood_alpha_i = alt_likelihood_alpha_i_pre + alt_likelihood_alpha_i_post
    dof = 2
    test_stat_i, p_val_i = likelihood_ratio_test(null_likelihood, 
                                        alt_likelihood_alpha_i, dof, log_pvals=0)
    curr_mat = symmetrize_from_vector(H_s[-1], dim)
    nonzero_cols = np.nonzero(np.any(curr_mat != 0, axis=0))[0]
    C_val = len(nonzero_cols)
    if C_val > 0:
        correction_factor = dim/C_val
        print(correction_factor)
        # correct p_vals for cluster at all time points independently
        p_val_i = p_val_i*correction_factor
    else:
        p_val_i = 1.0
    p_vals_zero.append(p_val_i)
    coeff_diffs_zero.append(alpha_i_change_pre[-1]-alpha_i_change_post[-1])
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  2  6 11 13 17 27 33 36 37]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  3  4  5  8  9 10 12 15 16 18 20 24 25 31 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 7 19 30 35]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [14 21 22 23 26 28 29 32 34 39]
****************************************

4.0
****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  2  6  8  9 12 13 17 24 26 29 37 38 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contai

In [215]:
sns.histplot(coeff_diffs_opo, kde=True)
plt.title("Difference: {}".format("0.1"))
plt.xlabel("Coefficient Difference")
plt.savefig('../lrt_test_figs/{}'.format("coeff_diffs_opo.png"))
plt.close()
sns.histplot(coeff_diffs_opf, kde=True)
plt.title("Difference: {}".format("0.4"))
plt.xlabel("Coefficient Difference")
plt.savefig('../lrt_test_figs/{}'.format("coeff_diffs_opf.png"))
plt.close()
sns.histplot(coeff_diffs_ope, kde=True)
plt.title("Difference: {}".format("0.8"))
plt.xlabel("Coefficient Difference")
plt.savefig('../lrt_test_figs/{}'.format("coeff_diffs_ope.png"))
plt.close()
sns.histplot(coeff_diffs_zero, kde=True)
plt.title("Difference: {}".format("0.0"))
plt.xlabel("Coefficient Difference")
plt.savefig('../lrt_test_figs/{}'.format("coeff_diffs_zero.png"))
plt.close()

In [216]:
sns.histplot(np.log(np.array(p_vals_opo)), kde=True)
plt.title("Difference: {}".format("0.1"))
plt.xlabel("log(Pvalue)")
plt.savefig('../lrt_test_figs/{}'.format("pvals_opo.png"))
plt.close()
sns.histplot(np.log(np.array(p_vals_opf)), kde=True)
plt.title("Difference: {}".format("0.4"))
plt.xlabel("log(Pvalue)")
plt.savefig('../lrt_test_figs/{}'.format("pvals_opf.png"))
plt.close()
sns.histplot(np.log(np.array(p_vals_ope)), kde=True)
plt.title("Difference: {}".format("0.8"))
plt.xlabel("log(Pvalue)")
plt.savefig('../lrt_test_figs/{}'.format("pvals_ope.png"))
plt.close()
sns.histplot(np.log(np.array(p_vals_zero)), kde=True)
plt.title("Difference: {}".format("0.0"))
plt.xlabel("log(Pvalue)")
plt.savefig('../lrt_test_figs/{}'.format("pvals_zero.png"))
plt.close()

In [200]:
plt.xticks([1, 2, 3, 4], ["0.0", "0.1", "0.4", "0.8"])
plt.xticks(rotation=45, ha='right')
ylim_max = 10.0
plot_confidence_interval(x=1, values=coeff_diffs_zero, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=2, values=coeff_diffs_opo, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=3, values=coeff_diffs_opf, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=4, values=coeff_diffs_ope, z=1.96, color='#2187bb', horizontal_line_width=0.25)
#plt.ylim(0, ylim_max)
plt.title("Confidence Intervals {}".format("Differences"))
plt.ylabel("Estimated Difference")
plt.xlabel("Coefficient Difference")
plt.tight_layout()
plt.savefig('../lrt_test_figs/{}'.format("conf_int_diff.png"))
plt.close()

In [218]:
plt.xticks([1, 2, 3, 4], ["0.0", "0.1", "0.4", "0.8"])
plt.xticks(rotation=45, ha='right')
ylim_max = 10.0
plot_confidence_interval(x=1, values=np.log(np.array(p_vals_zero)), z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=2, values=np.log(np.array(p_vals_opo)), z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=3, values=np.log(np.array(p_vals_opf)), z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=4, values=np.log(np.array(p_vals_ope)), z=1.96, color='#2187bb', horizontal_line_width=0.25)
#plt.ylim(0, ylim_max)
plt.title("Confidence Intervals {}".format("log(P Values)"))
plt.ylabel("log(P Value)")
plt.xlabel("Coefficient Difference")
plt.tight_layout()
plt.savefig('../lrt_test_figs/{}'.format("conf_int_pvals.png"))
plt.close()

In [223]:
import rpy2
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri
from rpy2.robjects.packages import importr
import rpy2.robjects.packages as rpackages
from rpy2.rinterface_lib.callbacks import logger as rpy2_logger
import logging
rpy2_logger.setLevel(logging.ERROR)

r = robjects.r
rpy2.robjects.numpy2ri.activate()
utils = importr('utils')
utils.chooseCRANmirror(ind=1)
# R package names
packnames = ('scalreg')

# R vector of strings
from rpy2.robjects.vectors import StrVector

# Selectively install what needs to be install.
# We are fancy, just because we can.
names_to_install = [x for x in packnames if not rpackages.isinstalled(x)]
if len(names_to_install) > 0:
    utils.install_packages(StrVector(names_to_install))
#clime = importr('clime')
scalreg = importr('scalreg')
from kesh_cpd import calc_T_t, calc_g1, calc_g2, calc_rhat

w = 100
def clime_init_fn(lam, data_minimal):
    nrow, ncol = data_minimal.shape
    X = r.matrix(data_minimal, nrow=nrow, ncol=ncol)
    reg_soln = scalreg.scalreg(X, lam0=lam)
    reg_soln_dict = dict(zip(reg_soln.names, list(reg_soln)))
    clime_est = reg_soln_dict['precision']
    # nrow, ncol = data_minimal.shape
    # X = r.matrix(data_minimal, nrow=nrow, ncol=ncol)
    # clime_out = clime.fastclime(X, self.lam, 100)
    # clime_soln = dict(zip(clime_out.names, list(clime_out)))
    # lambdamtx = clime_soln['lambdamtx']
    # icovlist = clime_soln['icovlist']
    # select_out = clime.fastclime_selector(lambdamtx, icovlist, self.lam)
    # select_soln = dict(zip(select_out.names, list(select_out)))
    
    # clime_est = np.array(select_soln['icov'])
    return clime_est
clime_init = clime_init_fn(5e-2, data_one.T)
print(clime_init.shape)
g1 = calc_g1(w)
g2 = calc_g2(w)
rhat0 = calc_rhat(clime_init, dim)
T0 = calc_T_t(X=data_two.T, omega_hat=clime_init, r_hat=rhat0, w=w, p=dim, t=0, g1=g1, g2=g2)


(40, 40)


In [224]:
T0

322.78967811825135

In [246]:
from scipy.stats import zscore
from scipy.stats import norm
def kesh_p_value(test_statistics):
    """
    Calculate z-score, then take survival function of normal dist
    """
    
    #z_scores = zscore(test_statistics)
    z_scores = (test_statistics - 0)/1 # substract mean and divide by standard deviation of true distribution?
    p_values = 1-norm.cdf(z_scores)
    
    return p_values

seed_list = np.arange(0, 100)
test_stats_opo_kesh = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.1
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    clime_init = clime_init_fn(5e-2, data_one.T)
    g1 = calc_g1(w)
    g2 = calc_g2(w)
    rhat0 = calc_rhat(clime_init, dim)
    T0 = calc_T_t(X=data_two.T, omega_hat=clime_init, r_hat=rhat0, w=w, p=dim, t=0, g1=g1, g2=g2)
    test_stats_opo_kesh.append(T0)
p_vals_opo_kesh = kesh_p_value(np.array(test_stats_opo_kesh))
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  4  9 12 15 20 22 24 26 32 33 34 37]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  2  3  6  7  8 10 14 17 18 19 23 25 28 29 31 36 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 5 13 16 21 27 30 35 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [11]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0 17 18 23 25 28 31 36]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  3  4  5  6  7

In [247]:
seed_list = np.arange(0, 100)
test_stats_opf_kesh = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.4
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    clime_init = clime_init_fn(5e-2, data_one.T)
    g1 = calc_g1(w)
    g2 = calc_g2(w)
    rhat0 = calc_rhat(clime_init, dim)
    T0 = calc_T_t(X=data_two.T, omega_hat=clime_init, r_hat=rhat0, w=w, p=dim, t=0, g1=g1, g2=g2)
    test_stats_opf_kesh.append(T0)
p_vals_opf_kesh = kesh_p_value(np.array(test_stats_opf_kesh))
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  3  5  9 17 21 23 25 26 31 35 37 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 1  2  6  8 16 20 22 28 29 32]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 4 10 15 18 24 27 30 33 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [ 7 11 12 13 14 19 34 36]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  4  5  6  9 12 13 14 15 17 22 25 28 30 34 35 36 38 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Chan

In [248]:
seed_list = np.arange(0, 100)
test_stats_ope_kesh = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.8
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    clime_init = clime_init_fn(5e-2, data_one.T)
    g1 = calc_g1(w)
    g2 = calc_g2(w)
    rhat0 = calc_rhat(clime_init, dim)
    T0 = calc_T_t(X=data_two.T, omega_hat=clime_init, r_hat=rhat0, w=w, p=dim, t=0, g1=g1, g2=g2)
    test_stats_ope_kesh.append(T0)
p_vals_ope_kesh = kesh_p_value(np.array(test_stats_ope_kesh))
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  1  3  5  6  7  8  9 10 12 13 15 19 22 23 27 28 31 34 35 37 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 2 16 17 18 20 26 29 30 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 4 11 21 24 32 36]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [14 25 33]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  1 10 14 16 17 19 20 23 25 27 29 30 31 32 33 37 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channel

In [249]:
seed_list = np.arange(0, 100)
test_stats_zero_kesh = []
for curr_seed in seed_list:
    H_s = generate_matrices_orthogonal(M=M, dim=dim)[0]
    prec_one = collect_precision_matrix(H_s=H_s, prec_coeffs=np.ones(M), P=dim)
    second_prec_coeffs = np.ones(M)
    second_prec_coeffs[-1] = second_prec_coeffs[-1] - 0.0
    prec_two = collect_precision_matrix(H_s=H_s, prec_coeffs=second_prec_coeffs, P=dim)
    data_one, _ = sim_data(covar=inv(prec_one), dim=dim, N=100)
    data_two, _ = sim_data(covar=inv(prec_two), dim=dim, N=100)
    data_full = np.concatenate((data_one, data_two), axis=1)
    clime_init = clime_init_fn(5e-2, data_one.T)
    g1 = calc_g1(w)
    g2 = calc_g2(w)
    rhat0 = calc_rhat(clime_init, dim)
    T0 = calc_T_t(X=data_two.T, omega_hat=clime_init, r_hat=rhat0, w=w, p=dim, t=0, g1=g1, g2=g2)
    test_stats_zero_kesh.append(T0)
p_vals_zero_kesh = kesh_p_value(np.array(test_stats_zero_kesh))
    

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  1  5  9 10 11 12 15 16 17 20 22 24 25 29 30 32 35 36 37 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Contained [ 2  3  4  6 21 23 26 27 33 34 39]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 2
Sim Channels Contained [ 7 13 19]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 3
Sim Channels Contained [ 8 14 18 28 31]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 0
Sim Channels Contained [ 0  2  3  7  9 13 17 18 26 27 29 33 35 36 37 38]
****************************************

****************************************
SIMULATION MATRICES
Sim Basis Matrix 1
Sim Channels Cont

In [250]:
plt.xticks([1, 2, 3, 4], ["0.0", "0.1", "0.4", "0.8"])
plt.xticks(rotation=45, ha='right')
ylim_max = 10.0
plot_confidence_interval(x=1, values=test_stats_zero_kesh, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=2, values=test_stats_opo_kesh, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=3, values=test_stats_opf_kesh, z=1.96, color='#2187bb', horizontal_line_width=0.25)
plot_confidence_interval(x=4, values=test_stats_ope_kesh, z=1.96, color='#2187bb', horizontal_line_width=0.25)
#plt.ylim(0, ylim_max)
plt.title("Confidence Intervals {}".format("Test Stat Kesh"))
plt.ylabel("Test Stat")
plt.xlabel("Coefficient Difference")
plt.tight_layout()
plt.savefig('../lrt_test_figs/{}'.format("conf_int_tstats_kesh.png"))
plt.close()

In [245]:
p_vals_ope_kesh

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])